In [0]:
from pyspark.sql import SparkSession, DataFrame
from datetime import datetime
from pyspark.sql.functions import to_timestamp, col, date_trunc, current_timestamp, to_date, concat_ws, sha2
from delta.tables import DeltaTable
import logging

In [0]:
def date_now() -> datetime:
    return datetime.strptime(datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "%Y-%m-%d %H:%M:%S")

In [0]:
dbutils.widgets.text("target_table_name", "", "Bronze name")
dbutils.widgets.text("source_table_name", "", "Stage name")
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"])

bronze_name = dbutils.widgets.get("target_table_name")
stage_name = dbutils.widgets.get("source_table_name")
environment = dbutils.widgets.get("environment")

In [0]:
source_path = f'`comercial-{environment}`.`stage`.`{stage_name}`'
target_path = f'`comercial-{environment}`.`bronze`.`{bronze_name}`'
logs_path = f'`comercial-{environment}`.`corporate`.`logs`'

In [0]:
spark = SparkSession.builder \
    .appName("Ingestion from csv to delta") \
    .getOrCreate()

logging.basicConfig(
    level=logging.INFO
    , format="%(asctime)s | %(levelname)s | %(message)s"
    , datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

In [0]:
def read_stage(stage_name: str) -> DataFrame: 
    df_stage = spark.table(source_path)
    return df_stage


In [0]:
def table_exists_in_catalog(table_name: str) -> bool:
    return spark.catalog.tableExists(table_name)

def get_delta_table(df_table_name: str) -> DeltaTable:
    return DeltaTable.forName(spark, df_table_name)

In [0]:
logs = {
    "dat_Hr_Process_Start": date_now(),
    "dat_Hr_Process_End":  '1900-01-01 00:00:00',
    "layer": "BRONZE",
    "catalog": f'comercial-{environment}'.upper(),
    "source_Name": stage_name.upper(),
    "target_Name": bronze_name.upper(),
    "status": None,
    "count_Rows": None,
}

In [0]:
def merge_stage_bronze_table(stage: DataFrame):
    if table_exists_in_catalog(target_path):
        get_delta_table(target_path) \
            .alias("target") \
            .merge(
                stage.alias("source"),
                "source.hash_column = target.hash_column"
            ) \
            .whenNotMatchedInsertAll() \
            .execute()
    else: 
        stage.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_path)

def save_logs(logs):
    logs.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(logs_path) 

In [0]:

try:
    logger.info("===========Start Process===========")
    logger.info("Start reading stage table")
    df_stage = read_stage(stage_name)
    logger.info("End reading stage table")


    logger.info("Start Ingestion Bronze")
    merge_stage_bronze_table(df_stage)
    logger.info("End Ingestion Bronze")
    

    logger.info("Start save logs")

    logs["status"] = "SUCCESS"
    logs["count_Rows"] = df_stage.count()
    logs["dat_Hr_Process_End"] = date_now()

    df_logs = spark.createDataFrame([logs])
    
    save_logs(df_logs)
    logger.info("End saving logs")
    logger.info("===========End Process===========")
except Exception as e:
    logger.error(f'{e}')